# Day 10 — Testing with pytest: Asserts, Fixtures, Parametrize, Testing Errors & Faking HTTP (3 Nov)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: learn to prove that code works with automated tests. You write tests with plain assert, read pytest's failure report, give each test fresh inputs with fixtures (including temporary folders), run one test over a whole table of cases with parametrize, prove that bad input fails the right way with pytest.raises, and test code that calls an API or reads environment variables without touching the network or your real settings.

- Deliverable: day10/test_contact_store.py and day10/test_cms_export.py, two green test suites for Day 4's contact_store.py and Day 6's cms_export.py, plus day10/dsa.py. day10/pytest.ini (given) lets the tests import the Day 4 and Day 6 modules.

- Where this sits: Week 2, Phase 1 (Foundation), day 3 of 7. Every later project ships with tests: FastAPI endpoints in Week 4, prompt and tool-use code in Phase 2, and the eval suites in Phase 4 are tests with an LLM in the loop. Today is the foundation for all of them.

Time split (5h):

| Block | Time | What you do |
|---|---|---|
| Theory | 1.5h | Concepts 1 to 6, typing every example in a scratch folder |
| Hands-on | 2.5h | Part A: contact_store tests, then the Day 4 code; Part B: cms_export tests |
| DSA | 0.5h | Valid palindrome, binary search |
| Revision | 0.5h | Quiz, Remember cells in the notebook, mutation check notes |

Open items first: the repo on GitHub still has only day01/ to day04/, and day04/contact_store.py and main.py are still the starter TODOs. Today is the best day to fix that, because tests turn "I think it works" into "it passes". Part A below has you write the tests for contact_store.py first and then finish the five functions until the tests go green. That is test-driven development (TDD), and it is how many teams work. If day06/cms_export.py is not on your machine, do Part A fully; Part B's file skips itself cleanly until cms_export.py exists.

Where to type today's examples: in a scratch folder outside the repo, for example ~/scratch/day10, so the demo tests never mix with your real suites. Install pytest into your venv first with python3 -m pip install pytest (version 9.x at the time of writing) and check it with python3 -m pytest --version.

## Recap (2 minutes, from memory)

- Predict the first block below: what ends up in report.txt, what ends up in warnings.txt, and which exit code is printed for the sample log?

- In the second block, does the script in tab 2 find the key? Why or why not, and which Python call reads it?

- In the command python3 -m pytest -q && git push, what happens to the push when one test fails, and why?

```bash
# Recap: predict, then run inside day09/ (answers at the end)
python3 log_report.py < server.log > report.txt 2> warnings.txt
echo "exit: $?"
wc -l < warnings.txt
# Two terminal tabs. Tab 1:
export CMS_API_KEY=demo-key-123
# Tab 2 (opened earlier), inside day06/:
python3 cms_export.py blog_post  # does it find the key?
```

Answers are on the last page of today's session.

## Concept 1: pytest, assert and reading a failure

### Why it matters for an FDE

At a customer you change code that other people depend on: an export their content team runs every night, a webhook that publishes entries, an integration that feeds their search. Every change can break something that worked yesterday. A test is a small program that runs your code with known inputs and checks the result, so the check runs in one second instead of you clicking through the flow by hand.

Tests are also evidence. When you open a pull request in a customer repo, a green test run is how a reviewer trusts a stranger's change. In FDE interviews, "how would you test this?" comes up in almost every coding round, and a candidate who writes tests without being asked stands out.

### How it works

pytest is the standard Python test runner, the way Jest is in JavaScript. You write ordinary functions whose names start with test_, in files whose names start with test_. pytest finds them (this is called discovery), runs each one, and reports which passed. Inside a test you check results with Python's own assert statement: assert x == 3 does nothing when it is true and raises an AssertionError when it is false. There is no expect(x).toBe(3) to learn; pytest rewrites your asserts so that a failure shows both sides of the comparison.

Here is the small module the examples test. Type it as cms_utils.py in your scratch folder. It has six functions, each of a kind you will test today: pure functions, a file reader, an environment setting, an HTTP call and a CLI entry point.

In [ ]:
# cms_utils.py: the code under test in today's examples
import json
import math
import os
import sys
import time
from pathlib import Path


class EntryError(ValueError):
    """Raised when a CMS entry or setting is unusable."""


def slugify(title: str) -> str:
    """'  Hello, World! ' -> 'hello-world' (a URL-safe slug)."""
    # keep letters and digits, turn everything else into a space
    cleaned = "".join(c if c.isalnum() else " " for c in title.lower())
    return "-".join(cleaned.split())


def reading_minutes(word_count: int, wpm: int = 200) -> int:
    """Minutes to read a post, rounded up, never less than 1."""
    if word_count < 0:
        raise EntryError(f"word_count must be >= 0, got {word_count}")
    return max(1, math.ceil(word_count / wpm))


def load_entries(path: Path) -> list[dict]:
    """Read a JSON list of entries; [] if the file does not exist."""
    try:
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        return []
    if not isinstance(data, list):
        raise EntryError(f"{path.name} must hold a JSON list")
    return data


def get_base_url() -> str:
    """Read CMS_BASE_URL from the environment, without a trailing /."""
    url = os.environ.get("CMS_BASE_URL", "").strip()
    if not url:
        raise EntryError("CMS_BASE_URL is not set")
    return url.rstrip("/")


def fetch_title(session, base_url: str, uid: str) -> str:
    """GET one entry's title; on a 503 wait 2 s and try once more."""
    url = f"{base_url}/entries/{uid}"
    r = session.get(url, timeout=5)
    if r.status_code == 503:  # busy right now: wait, then retry once
        time.sleep(2)
        r = session.get(url, timeout=5)
    if r.status_code != 200:
        raise EntryError(f"HTTP {r.status_code} for {uid}")
    return r.json()["entry"]["title"].strip()


def main(argv: list[str]) -> int:
    """CLI: print the slug of the words given; return an exit code."""
    if not argv:
        print("error: give a title", file=sys.stderr)
        return 2
    print(slugify(" ".join(argv)))
    return 0

Good tests follow the same three steps, called Arrange, Act, Assert: set up the input, call the code once, check the result. Each test checks one behaviour and its name says which, so a failing name in the report already tells you what broke.

```bash
# test_basics.py: pytest finds this file because its name starts with test_
from cms_utils import reading_minutes, slugify


def test_slugify_lowercases_and_joins_words():
    # Arrange: the input a content editor would type
    title = "  Getting Started with SSO "
    # Act: call the code under test once
    result = slugify(title)
    # Assert: a plain assert; pytest shows both sides if it fails
    assert result == "getting-started-with-sso"


def test_slugify_drops_punctuation():
    title = 'Pricing FAQ: "Starter" vs "Pro"'
    assert slugify(title) == "pricing-faq-starter-vs-pro"


def test_reading_minutes_rounds_up():
    assert reading_minutes(201) == 2  # 1.005 minutes rounds UP to 2


def test_reading_minutes_is_at_least_one():
    assert reading_minutes(0) == 1

# --- run it, in the terminal, inside the same folder ---
python3 -m pytest test_basics.py -v
# -> collected 4 items
# -> test_basics.py::test_slugify_lowercases_and_joins_words PASSED [ 25%]
# -> test_basics.py::test_slugify_drops_punctuation PASSED          [ 50%]
# -> test_basics.py::test_reading_minutes_rounds_up PASSED          [ 75%]
# -> test_basics.py::test_reading_minutes_is_at_least_one PASSED    [100%]
# -> ============ 4 passed in 0.02s ============
```

Always run pytest as python3 -m pytest rather than plain pytest. The -m form uses the same Python (and the same venv) as python3, which avoids Day 9's "installed it, still ModuleNotFoundError" problem, and it also adds the current folder to the import path.

Now make a test fail on purpose and learn to read the report. This is the skill you use most: the report tells you which test, which line, what you expected and what the code returned.

In [ ]:
# test_fail_demo.py: a test that fails ON PURPOSE, to read the report
from cms_utils import slugify


def test_slug_keeps_accents():
    # we EXPECT the accents to be dropped, but isalnum() keeps them
    assert slugify("Renée's café") == "renees-cafe"

# --- python3 -m pytest test_fail_demo.py prints (shortened) ---
# -> >       assert slugify("Renée's café") == "renees-cafe"
# -> E       AssertionError: assert 'renée-s-café' == 'renees-cafe'
# -> E         - renees-cafe
# -> E         + renée-s-café
# -> test_fail_demo.py:7: AssertionError
# -> FAILED test_fail_demo.py::test_slug_keeps_accents
# -> ============ 1 failed in 0.02s ============
# The > line is the failing assert; E lines show BOTH values.
# "-" is what you expected, "+" is what the code really returned.

Here the test found a real gap: the slug keeps accented letters, which some CMS URL rules reject. A failing test is either a bug in the code or a wrong expectation in the test. Decide which before you "fix" anything. For this module we keep the behaviour and correct the test, because Unicode slugs are valid in Contentstack.

### Real-world FDE example

A customer's Drupal-to-Contentstack migration script turns 40,000 article titles into URL slugs. Two slugs that collide silently overwrite each other's entries. A ten-line test file with the tricky titles from their real data (accents, emoji, all punctuation, very long titles) runs on every change, so nobody can break the slug rules three weeks later without the test going red first.

### Common mistakes

- Naming the function check_slug or the file slug_tests.py. pytest only collects test_* functions in test_*.py files, so the test is silently never run.

- Testing many behaviours in one long test. The first failing assert stops the test, so you see one problem and miss the rest. Write one behaviour per test.

- Printing results and looking at them instead of asserting. A print never fails; only an assert makes the test red.

- Writing the expected value by running the code and copying its output into the test. Then the test only proves the code does what it does. Work out the expected value by hand from the requirement.

## Concept 2: Fixtures, tmp_path and conftest.py

### Why it matters for an FDE

Most real code needs something prepared before it runs: a file on disk, a settings dict, a database with sample rows, a client with a fake key. If every test builds that by hand, tests get long and copy-pasted. Worse, if tests share one real file, one test's leftovers break the next one, and a test that passes alone fails in the full run. Fixtures solve both problems.

### How it works

A fixture is a function marked with @pytest.fixture that prepares a value. A test asks for it by putting the fixture's name in its own parameter list, and pytest calls the fixture and passes in the result. You never call a fixture yourself. This is the same idea as React context or dependency injection: the test declares what it needs and the framework supplies it.

pytest ships with useful fixtures. The most important today is tmp_path: a brand-new empty folder (a pathlib Path, Day 4) for each test, deleted automatically later. Every test that writes files should write inside tmp_path, never into your real day04/data/.

In [ ]:
# test_fixtures.py: fixtures give each test fresh, ready-made inputs
import json

import pytest

from cms_utils import load_entries


@pytest.fixture  # marks a function as a fixture; tests ask for it by NAME
def entries_file(tmp_path):
    # tmp_path is a built-in fixture: a new empty folder for EACH test
    path = tmp_path / "entries.json"
    data = [{"uid": "blt001", "title": "SSO"}, {"uid": "blt002"}]
    path.write_text(json.dumps(data), encoding="utf-8")
    return path  # the test receives this value as its argument


def test_load_entries_reads_list(entries_file):
    entries = load_entries(entries_file)
    assert [e["uid"] for e in entries] == ["blt001", "blt002"]


def test_missing_file_gives_empty_list(tmp_path):
    assert load_entries(tmp_path / "nope.json") == []


def test_each_test_gets_its_own_folder(tmp_path, entries_file):
    # entries_file used the SAME tmp_path as this test (one per test)
    print(tmp_path.name, sorted(p.name for p in tmp_path.iterdir()))
    assert entries_file.parent == tmp_path

# --- python3 -m pytest test_fixtures.py -v -s  (-s shows print output) ---
# -> test_fixtures.py::test_load_entries_reads_list PASSED
# -> test_fixtures.py::test_missing_file_gives_empty_list PASSED
# -> test_fixtures.py::test_each_test_gets_its_own_folder
# ->   test_each_test_gets_its_own_fo0 ['entries.json']  PASSED
# -> ============ 3 passed in 0.03s ============

Two details in that output matter. Each test got its own folder (the last test sees only the one file its fixture wrote), and a fixture can use another fixture: entries_file asked for tmp_path, and the test that asked for both got the same folder. Within one test, each fixture runs once and its value is shared.

When setup needs a matching cleanup (close a connection, delete a record you created in a sandbox CMS), use yield instead of return. Code before yield is setup, code after it is teardown, and the teardown runs even when the test fails, like beforeEach and afterEach in Jest. Put fixtures that several test files need in a file called conftest.py; pytest loads it automatically, with no import.

In [ ]:
# conftest.py: fixtures here are shared by every test file in this folder
import pytest


@pytest.fixture
def cms_settings():
    # code BEFORE yield is setup, like beforeEach in Jest
    settings = {"base_url": "https://cms.test", "api_key": "test-key"}
    print("\n  setup: settings ready")
    yield settings  # the test runs here, using this value
    # code AFTER yield is teardown, like afterEach; runs even if it failed
    settings.clear()
    print("  teardown: settings cleared")

# test_conftest_demo.py: uses a fixture from conftest.py, no import needed
def test_settings_have_key(cms_settings):
    print("  test body runs")
    assert cms_settings["api_key"] == "test-key"

# --- python3 -m pytest test_conftest_demo.py -s -q ---
# ->   setup: settings ready
# ->   test body runs
# -> .  teardown: settings cleared
# -> 1 passed in 0.01s

### Real-world FDE example

For a Contentstack webhook handler you write a webhook_payload fixture that loads a real (anonymised) "entry published" payload from a JSON file. Twenty tests use it: one changes the locale, another removes the author, another sends an unknown content type. When Contentstack adds a field to the payload, you update one fixture file instead of twenty tests.

### Common mistakes

- Calling the fixture like a function inside the test (entries_file()). pytest refuses that with an error. Put the name in the test's parameters instead.

- Writing to a fixed path such as data/contacts.json in tests. It overwrites your real data and makes tests depend on each other. Use tmp_path.

- Returning a mutable object from a shared fixture and changing it in one test. With the default scope each test gets a fresh one, which is what you want; only widen a fixture's scope (scope="module" or "session") for slow, read-only setup.

- Doing the cleanup after a return. Nothing after return runs; use yield.

## Concept 3: Parametrize, one test over a table of cases

### Why it matters for an FDE

Validation code (emails, slugs, dates, status codes) has many small cases, and the bugs live at the edges: an empty string, exactly 200 words, a trailing slash. Writing a separate test function for each case is tedious, so people skip cases. Parametrize turns the cases into a table that is quick to extend, which is how you end up actually testing the edges.

### How it works

@pytest.mark.parametrize("a, b", [...]) takes the argument names as one string and a list of tuples. pytest runs the test once per tuple and reports each run as a separate test, with the values in its name. It is test.each in Jest. Add ids= to give each case a readable name, which helps with -k.

```bash
# test_param.py: one test function, many input/expected pairs
import pytest

from cms_utils import reading_minutes, slugify


@pytest.mark.parametrize(
    "title, expected",  # names of the arguments, as one string
    [
        ("Hello World", "hello-world"),
        ("  spaces   everywhere ", "spaces-everywhere"),
        ("Q4 2026: roadmap!", "q4-2026-roadmap"),
        ("---", ""),  # edge case: nothing left after cleaning
    ],
)
def test_slugify_cases(title, expected):
    assert slugify(title) == expected


@pytest.mark.parametrize(
    "words, minutes",
    [(0, 1), (1, 1), (200, 1), (201, 2), (1000, 5)],
    ids=["zero", "one", "exact", "just-over", "long"],  # readable names
)
def test_reading_minutes_boundaries(words, minutes):
    assert reading_minutes(words) == minutes

# --- python3 -m pytest test_param.py -v ---
# -> collected 9 items         (4 slug cases + 5 minute cases)
# -> test_slugify_cases[Hello World-hello-world] PASSED
# -> test_slugify_cases[  spaces   everywhere -spaces-everywhere] PASSED
# -> test_slugify_cases[Q4 2026: roadmap!-q4-2026-roadmap] PASSED
# -> test_slugify_cases[----] PASSED
# -> test_reading_minutes_boundaries[zero] PASSED
# -> ...[one], [exact], [just-over], [long] PASSED
# -> ============ 9 passed in 0.02s ============
# --- run only some cases by name with -k ---
python3 -m pytest test_param.py -q -k "just-over or spaces"
# -> 2 passed, 7 deselected in 0.01s
```

Pick cases deliberately: one normal case, then the boundaries (0, 1, exactly the limit, one over it), then the odd inputs (empty, only spaces, only punctuation). For reading_minutes, 200 and 201 words are the cases that catch a wrong rounding rule; ten "normal" cases would not.

### Real-world FDE example

A customer's CMS export has dates in four formats because three teams entered content over eight years. Your parse_date gets a parametrize table with one row per format seen in their data, plus the bad ones you must reject. When their team finds a fifth format next month, the fix is one new row and one code change, and the table documents every format you support.

### Common mistakes

- Naming the arguments in the string differently from the test's parameters ("title, expected" but def test_x(title, result)). pytest stops with an error that names the missing argument.

- Putting logic in the test to compute the expected value (expected = title.lower().replace(...)). That repeats the code under test, bugs included. Write the expected value as a literal.

- Stacking two parametrize decorators without noticing that they multiply: 3 cases times 2 cases makes 6 tests.

## Concept 4: Testing errors and output with pytest.raises and capsys

### Why it matters for an FDE

Half of integration code is about failure: a missing key, a broken file, a 401 from the customer's API. Day 4 and Day 6 taught you to raise clear, specific errors. Today you prove they happen, with the right type and the right message, because a customer's on-call engineer will read that message at 2 a.m. A CLI's output and exit code are also part of its contract: cron and CI depend on them (Day 9).

### How it works

with pytest.raises(SomeError): passes only if the code inside the block raises that exception or a subclass of it; if nothing is raised, the test fails with "DID NOT RAISE". Add match="..." to also check the message. match is a regular expression search, so plain words work, but characters such as ., (, [ and ? have special meanings; wrap literal text in re.escape() when it contains them. Write as exc_info to keep the error object and inspect it after the block.

The built-in capsys fixture captures everything printed to stdout and stderr during the test. capsys.readouterr() returns both as strings, so you can check a CLI's output and that errors went to stderr, not stdout.

In [ ]:
# test_errors.py: proving that bad input fails the RIGHT way
import json

import pytest

from cms_utils import EntryError, load_entries, main, reading_minutes


def test_negative_word_count_raises():
    # passes only if the block raises EntryError (or a subclass of it)
    with pytest.raises(EntryError, match="must be >= 0, got -5"):
        reading_minutes(-5)


def test_error_object_can_be_inspected(tmp_path):
    path = tmp_path / "entries.json"
    path.write_text(json.dumps({"uid": "blt001"}), encoding="utf-8")
    with pytest.raises(EntryError) as exc_info:  # keep the error to look at
        load_entries(path)
    assert "entries.json" in str(exc_info.value)
    # EntryError inherits from ValueError (Day 5), so this is True too
    assert isinstance(exc_info.value, ValueError)


def test_main_prints_slug_and_returns_0(capsys):
    code = main(["Hello", "World"])
    out, err = capsys.readouterr()  # everything printed so far, split in two
    assert (code, out, err) == (0, "hello-world\n", "")


def test_main_without_args_returns_2(capsys):
    assert main([]) == 2
    assert capsys.readouterr().err == "error: give a title\n"

# --- python3 -m pytest test_errors.py -q ---
# -> 4 passed in 0.02s
# If the code does NOT raise, pytest.raises itself fails the test:
# -> Failed: DID NOT RAISE ValueError

Notice that main returns its exit code instead of calling sys.exit() itself; the sys.exit(main()) line sits under if __name__ == "__main__": (Day 6). That design makes main easy to test. If a function does call sys.exit(2), test it with pytest.raises(SystemExit) and check exc_info.value.code == 2.

### Real-world FDE example

Your Contentstack export must exit with 2 and the message "CMS_API_KEY is not set" when the key is missing, and with 1 on a 401. The customer's scheduler alerts a human only on non-zero exits, and their runbook tells the on-call engineer what each message means. Tests that pin the messages and codes stop a later refactor from quietly changing the contract the runbook depends on.

### Common mistakes

- Catching the error yourself with try/except in the test and asserting inside the except. If nothing is raised, the test passes anyway. pytest.raises fails when nothing is raised.

- Putting several lines inside the with pytest.raises block. If an earlier line raises the same error type for another reason, the test passes for the wrong reason. Keep only the one call that should fail inside the block.

- Using pytest.raises(Exception). Almost anything matches, including a typo that raises NameError. Name the specific error class.

- Forgetting the newline: print adds "\n", so the captured output is "hello-world\n".

## Concept 5: Faking the outside world with monkeypatch and fake objects

### Why it matters for an FDE

Your code talks to things you do not control: the customer's CMS, an LLM API, environment variables, the clock. A test that calls the real CMS is slow, fails when the Wi-Fi drops or the sandbox is down, can change real data, and cannot produce a 503 on demand. A test suite must be fast, repeatable and runnable offline in CI. So you replace the outside world with a fake that behaves exactly as you script it.

### How it works

The built-in monkeypatch fixture changes things for one test and undoes every change afterwards. monkeypatch.setenv("NAME", "value") and monkeypatch.delenv("NAME", raising=False) control environment variables. monkeypatch.setattr(obj, "name", new_value) replaces an attribute, for example time.sleep, so retry code does not really wait. It is jest.spyOn with automatic restore.

In [ ]:
# test_env.py: monkeypatch changes the world for ONE test, then undoes it
import pytest

import cms_utils
from cms_utils import EntryError, get_base_url


def test_base_url_strips_trailing_slash(monkeypatch):
    monkeypatch.setenv("CMS_BASE_URL", "https://cms.test/")  # set for 1 test
    assert get_base_url() == "https://cms.test"


def test_missing_base_url_is_a_clear_error(monkeypatch):
    # raising=False: do not complain if the variable was never set
    monkeypatch.delenv("CMS_BASE_URL", raising=False)
    with pytest.raises(EntryError, match="CMS_BASE_URL is not set"):
        get_base_url()


def test_sleep_can_be_replaced(monkeypatch):
    calls = []
    # replace time.sleep AS SEEN BY cms_utils with a fake that only records
    monkeypatch.setattr(cms_utils.time, "sleep", calls.append)
    cms_utils.time.sleep(2)  # returns at once, nothing waits
    assert calls == [2]

# --- python3 -m pytest test_env.py -q ---
# -> 3 passed in 0.01s

GOTCHA: patch the name where the code under test looks it up. cms_utils does import time and calls time.sleep, so patching cms_utils.time.sleep works. If a module did from time import sleep, it holds its own name sleep, and you would patch module.sleep instead.

For HTTP the cleanest approach is dependency injection: the function receives the session as a parameter (Day 6's fetch_page(session, ...) already does), so a test can pass in a fake. The fake only needs the parts your code uses: get() returning an object with status_code, json() and, for Retry-After, headers. Because the fake records each call, the test can also assert what was sent: the URL, the parameters and that a timeout was given.

In [ ]:
# test_http.py: test HTTP code with a FAKE session, no network at all
import pytest

import cms_utils
from cms_utils import EntryError, fetch_title


class FakeResponse:
    """Just the parts of requests.Response our code uses."""

    def __init__(self, status_code, body=None):
        self.status_code = status_code
        self._body = body or {}

    def json(self):
        return self._body


class FakeSession:
    """Plays back prepared responses in order and records each call."""

    def __init__(self, *responses):
        self.responses = list(responses)
        self.calls = []  # (url, kwargs) for every get(), to assert on later

    def get(self, url, **kwargs):
        self.calls.append((url, kwargs))
        return self.responses.pop(0)  # next prepared reply, like a script


@pytest.fixture
def no_sleep(monkeypatch):
    waits = []
    monkeypatch.setattr(cms_utils.time, "sleep", waits.append)
    return waits  # the test can check how long the code WOULD have waited


def test_returns_stripped_title():
    s = FakeSession(FakeResponse(200, {"entry": {"title": " SSO guide "}}))
    assert fetch_title(s, "https://cms.test", "blt001") == "SSO guide"
    url, kwargs = s.calls[0]
    assert url == "https://cms.test/entries/blt001"
    assert kwargs["timeout"] == 5  # a test can enforce "always a timeout"


def test_retries_once_on_503(no_sleep):
    s = FakeSession(FakeResponse(503),
                    FakeResponse(200, {"entry": {"title": "SSO"}}))
    assert fetch_title(s, "https://cms.test", "blt001") == "SSO"
    assert len(s.calls) == 2 and no_sleep == [2]  # waited 2 s, but not really


def test_404_fails_fast_without_retry(no_sleep):
    s = FakeSession(FakeResponse(404))
    with pytest.raises(EntryError, match="HTTP 404 for blt999"):
        fetch_title(s, "https://cms.test", "blt999")
    assert len(s.calls) == 1 and no_sleep == []  # no retry, no wait

# --- python3 -m pytest test_http.py -v ---
# -> test_http.py::test_returns_stripped_title PASSED         [ 33%]
# -> test_http.py::test_retries_once_on_503 PASSED            [ 66%]
# -> test_http.py::test_404_fails_fast_without_retry PASSED   [100%]
# -> ============ 3 passed in 0.02s ============

These three tests prove behaviour that is hard to check by hand: one retry after a 503, no retry on a 404, a timeout on every request. They run in milliseconds and never touch a network. Libraries such as responses (for requests) and respx (for httpx, which you meet in Day 13) intercept HTTP calls at a lower level, and unittest.mock builds fakes automatically. Hand-written fakes come first because you can see every line of what they do.

### Real-world FDE example

Before a go-live, a customer asks: "What happens if our CMS rate-limits you during the nightly sync?" Instead of waiting for it to happen in production, you show a test where the fake session returns 429 with Retry-After: 30, and the assertions prove the sync waits 30 seconds (faked), retries, and finishes. That answer, backed by a green test, is what earns trust on a technical call.

### Common mistakes

- Letting a test call the real API "just for now". It becomes the flaky test everyone learns to ignore. Real-API checks belong in a separate, clearly named smoke test, run on purpose.

- Changing os.environ directly in a test. The change leaks into every later test. Use monkeypatch.setenv.

- Making the fake smarter than the real thing, for example a fake that never returns an error. Script the failures you care about: 401, 429, 503, a timeout.

- Asserting only the return value. For HTTP code, also assert the calls: how many requests, which parameters, and that a timeout was set.

## Concept 6: Test layout, running tests and trusting them

### Why it matters for an FDE

In a customer repo you will run their tests long before you write your own, and CI runs yours on every push. Knowing where tests live, how they import the code, and how to run exactly the test you need saves hours. And a suite is only worth something if it would catch a real bug.

### How it works

pytest reads its settings from pytest.ini (or a [tool.pytest.ini_options] section in pyproject.toml, Day 12). The folder holding that file becomes the rootdir. The pythonpath setting adds folders to the import path, which is how day10's tests can import contact_store from day04/. In a real project the code lives in a package and tests in a tests/ folder; Day 12 moves cms_export into that shape.

```bash
# day10/pytest.ini: settings pytest reads when you run it on day10/
[pytest]
# folders to put on the import path, so "import contact_store" works
pythonpath = ../day04 ../day06
# test files to look for (this is also pytest's default)
python_files = test_*.py

# --- running tests, from the repo root ---
python3 -m pytest day10            # every test file in day10/
python3 -m pytest day10 -v         # one line per test, with its name
python3 -m pytest day10 -x         # stop at the first failure
python3 -m pytest day10 --lf       # rerun only the tests that failed last
python3 -m pytest day10 -k merge   # only tests whose name contains merge
python3 -m pytest day10 --co -q    # list the tests, run nothing
# one single test, by its "node id" (file::function)
python3 -m pytest day10/test_contact_store.py::test_merge_skips_dupes

# --- exit codes, Day 9 style ---
python3 -m pytest -q; echo "exit: $?"
# -> exit: 0 when all passed, 1 when any test failed,
# -> exit: 5 when no tests were collected (a typo in a file name?)
python3 -m pytest -q && git push   # push only if the suite is green
```

To check that your tests can catch bugs, break the code on purpose. Remove one .lower(), change start=1 to start=0, or make the retry code retry a 401, then run the suite. At least one test must go red; if none does, you have found a hole in your tests. Put the code back afterwards. This is a hand-made version of mutation testing, and it is today's last acceptance check.

### Real-world FDE example

On day one in a customer's repo you clone it, create a venv, install the requirements and run python3 -m pytest -q. If it is green you have a safe baseline, and any red you see after your change is yours. If it is already red before you touch anything, you write that down and tell the customer at once, so their broken tests are not blamed on you later.

### Common mistakes

- Fixing import errors in tests by adding sys.path.append(...) to every test file. Configure the path once, in pytest.ini or pyproject.toml.

- Tests that pass only in a certain order, because one test leaves a file or variable that the next one needs. Every test must pass alone: run it by its node id to check.

- Chasing a coverage number. A test that runs a line without asserting anything about it raises coverage and catches nothing. Assert behaviour.

- Running pytest from the wrong folder and getting "no tests ran" (exit code 5). Check where you are with pwd (Day 9).

## Check your understanding (predict first, then run)

Type this file into your scratch folder next to cms_utils.py. Write down a prediction for each check before you run it.

In [ ]:
# test_check.py: predict what pytest reports for each test, then run it
import os

import pytest

from cms_utils import reading_minutes


def check_slug():  # Check 1: is this run?
    assert False


def test_float_sum():  # Check 2: pass or fail?
    assert 0.1 + 0.2 == 0.3


def test_parent_class():  # Check 3: pass or fail?
    with pytest.raises(ValueError):
        reading_minutes(-1)


@pytest.mark.parametrize("a", [1, 2, 3])
@pytest.mark.parametrize("b", ["x", "y"])
def test_grid(a, b):  # Check 4: how many tests does this make?
    assert a and b


def test_set_env(monkeypatch):  # Check 5a
    monkeypatch.setenv("DAY10_DEMO", "on")
    assert os.environ["DAY10_DEMO"] == "on"


def test_read_env():  # Check 5b: pass or fail? (runs after 5a)
    assert os.environ.get("DAY10_DEMO") is None

# --- run it: python3 -m pytest test_check.py -q, then -v ---

- Check 1: is check_slug run? What would you change so it is?

- Check 2: does test_float_sum pass? If not, why, and what is the fix?

- Check 3: reading_minutes raises EntryError, but the test expects ValueError. Pass or fail, and why?

- Check 4: how many tests does test_grid produce?

- Check 5: does test_read_env pass, even though test_set_env set DAY10_DEMO just before it?

- Without running it: how many tests does pytest report for the whole file, and how many fail?

## Hands-on task (about 2.5 hours): day10 test suites

The scenario: a customer is about to run your Day 4 contact import and Day 6 CMS export on their own servers. Their tech lead says: "We run nothing in production that does not come with tests. Show me a green suite that does not need our network." Today you write those suites.

Before you start: git switch main && git pull, then git switch -c day10-tests. The starters are in day10/: pytest.ini (given, complete), test_contact_store.py, test_cms_export.py and dsa.py. Every test function in the starters has a comment saying what to prove and HINT lines; its body is raise NotImplementedError("TODO"), so it is red until you write it.

Part A, test_contact_store.py, then finish contact_store.py (110 min). Write the tests first, from the rules in contact_store.py's comments, and run them: they fail with NotImplementedError from contact_store. Then implement the five Day 4 functions one at a time until each group goes green: validate_contact, merge_contacts, save and load, then import. Finally run day04/main.py and check Day 4's acceptance criteria. The tests use the real day04/contacts_import.json for the import test and tmp_path for every file they write.

Part B, test_cms_export.py (40 min). Tests for entry_to_row, write_csv, get_api_key, fetch_page, fetch_all_entries and main, with a FakeSession like Concept 5's. Match the fakes to your own cms_export.py: look at how your code calls session.get and which response fields it reads. The file starts with pytest.importorskip("cms_export"), so it skips cleanly if day06/cms_export.py is missing.

```bash
# Hands-on target: from the repo root, when both parts are done
python3 -m pytest day10 -v
# -> collected 30+ items       (your exact count will differ)
# -> day10/test_contact_store.py::test_validate_cleans_good_record PASSED
# -> ...
# -> day10/test_cms_export.py::test_fetch_page_retries_503 PASSED
# -> ============ 30 passed in 0.15s ============
python3 -m pytest day10 --durations=3   # no test should take 0.1 s+
python3 -m pytest day10 -q; echo "exit: $?"
# -> exit: 0
```

Acceptance criteria:

- From the repo root, python3 -m pytest day10 -v shows every test passing: 0 failed and 0 skipped (a skip means cms_export.py is missing, so Part B is not done yet).

- python3 -m pytest day10 --co -q lists at least 15 tests in test_contact_store.py and at least 12 in test_cms_export.py. Parametrized cases count separately.

- Each file defines at least one fixture of your own, at least one parametrize, and at least two pytest.raises checks with match=.

- No network: the suite passes with mock_cms.py stopped and Wi-Fi off. No real waiting: --durations=3 shows no test at 0.1 s or slower.

- Tests never touch your real files: day04/data/contacts.json and day06/output/ are unchanged after a full run (check with git status and ls -l). Every write goes to tmp_path.

- Every test passes alone. Run two of them by node id to check.

- Mutation check: make each of these bugs on purpose, one at a time, and confirm at least one test goes red: remove the email .lower() in validate_contact; change start=1 to start=0 in import_contacts; add 401 to the statuses fetch_page retries. Put the code back. Note in the pull request which test caught each bug.

- day04/main.py now meets Day 4's acceptance criteria 1 to 7 (the tests helped you get there).

- The work goes in through the branch day10-tests in at least three commits (contact_store tests; contact_store implementation; cms_export tests), pushed, with a pull request that has What, Why, How to test and Open items.

Hints (no solutions, try first):

- Write one test, run it, watch it fail for the right reason, then write the next one. python3 -m pytest day10 -x stops at the first failure so the output stays short.

- For "does not change its input", keep a copy before the call with copy.deepcopy(raw) (or a hand-written dict literal) and compare after.

- The import test needs the path of the real export file. Path(contact_store.__file__).parent is the day04/ folder, wherever you run pytest from.

- For the broken-store test, write a file with invalid JSON into tmp_path (for example a JSON object with its closing brace missing) and check the message names the file and a line number, using match=.

- For fetch_page's connection error, a fake session can raise instead of return: store an exception instance in the prepared list, and in get(), raise it if isinstance(reply, Exception). Use requests.ConnectionError() so your code's except clause catches it.

- If fetch_page waits with time.sleep, patch cms_export.time.sleep (Concept 5's GOTCHA tells you which name to patch).

- To test main without HTTP, monkeypatch cms_export.fetch_all_entries with a small function that returns two hand-written entries, then check the return code and the CSV in tmp_path.

- If a test passes before you have written the code it tests, it is not testing anything. Expect red first.

## DSA practice (30 min)

Both problems are in day10/dsa.py with signatures, examples and hints. Today, also write a parametrized test for each in day10/test_dsa.py (stretch, not counted in the 15 + 12).

- Valid palindrome. is_palindrome("A man, a plan, a canal: Panama") is True: ignoring case and everything that is not a letter or digit, it reads the same both ways. Hint: either build the cleaned string (the slugify idea from Concept 1) and compare it with its reverse, or walk two indexes inward from both ends and skip characters that do not count. Ask yourself what an empty string should return.

- Binary search. binary_search([1, 3, 5, 8, 13], 8) returns 3, the index, and -1 when the target is missing. The list is sorted. Hint: keep low and high indexes, look at the middle one, and throw away the half that cannot contain the target. Your tests should include an empty list, a one-item list, and the first and last items: off-by-one bugs live there.

## Quiz (3 quick questions)

- Your teammate's test for the CMS export calls the customer's real sandbox API. It passes on their laptop and fails in CI about one run in five. Give two reasons this can happen and describe how you would rewrite the test.

- What is the difference between a fixture that uses return and one that uses yield? Give one FDE situation where you need yield.

- A customer's tech lead asks: "Your tests are all green. How do I know they test anything?" Give two concrete answers.

## What you achieved today

You can now write and run pytest suites, read a failure report, build clean inputs with fixtures and temporary folders, cover the edge cases of validation code with parametrize, prove that errors, messages, output and exit codes are right, and test code that depends on environment variables, sleeping and HTTP without any network. You also finished (or are about to finish) Day 4's store, which the repo has been waiting for. From here on, every task in the plan comes with tests.

Tomorrow (Day 11): iterators, generators, decorators and context managers, used to build a retry decorator and a streaming reader for a large JSONL CMS export.

## When you're done

Paste into the Forward Deployment Engineer project chat as "Day 10 review": your recap and check-your-understanding predictions next to the real outputs, test_contact_store.py, test_cms_export.py, the finished contact_store.py, the output of python3 -m pytest day10 -v, the three mutation results, dsa.py, the link to the pull request, and your quiz answers. Say which concept felt least natural; tomorrow's recap starts there.

## Recap answers

*Your answers:*
